In [5]:
import numpy as np
import math
import warnings
from scipy import optimize 
import warnings
import pandas as pd

In [16]:
def extractCosineTideParams(DMY, velSigned): 
    velMax_guess = []
    valMax = []
    indMax = []
    phi0_guess = [] 
    
    #remove NaN values 
    if (DMY.dtype == '<M8[ns]'): 
        for i in reversed(range(len(DMY))): 
            if math.isnan(velSigned[i]):
                del DMY[i] 
                del velSigned[i]

    #time converted to seconds from start
    t = (DMY - DMY[0]).astype('timedelta64[s]')

    #tide's period fixed as semidiurnal (12 hrs + 25 mins) according to 
    #https://oceanservice.noaa.gov/facts/tidefrequency.html)
    T_fixed = 12*3600 + 25*60

    #guess made based on maxima in signed velocity 
    velMax_guess = max(abs(velSigned))
    valMax = max(velSigned)
    indMax = np.argmax(valMax) 
    phi0_guess = t[indMax]
    
    # for i in range(len(velSigned)): 
    #     velMax_guess.append(max(abs(velSigned[i]))) 
    #     valMax.append(max(velSigned[i])) 
    #     indMax.append(velSigned[i].index(valMax[i]))  

    #     phi0_guess.append(t[indMax[i]]) 

    velMax_size = (np.array(velMax_guess)).shape
    phi0_size = (np.array(phi0_guess)).shape

    if velMax_size == phi0_size: 
        pass
        #print(""No transpose needed to correct array dimensions between maximum signed velocities and phi0.\n") 
    elif (velMax_size[0] == phi0_size[1]) and (velMax_size[1] == phi0_size[0]): 
        #print("Transpose needed to correct array dimensions between maximum signed velocities and phi0.\n")
        phi0_guess = (np.array(phi0_guess)).T #or np.tranpose()     
    else: 
        raise ValueError("The array's length of maximum signed velocities (%i) is not equal to that of phi0 (%i).\n" %(max(velMax_guess),max(phi0_guess)))

    #fitting to cosine by least squares 
    paramsGuess = [velMax_guess, phi0_guess] 

    def cosineFit(params, t): 
        y = params[0] * math.cos((2 * math.pi * (t - params(2)) ) / T_fixed - math.pi / 2) 
        return y 
        
    boundVal = ([0, 0], [2*velMax_guess, max(t)])

    try: 
        #replaced lsqcurvefit
        paramsFit, covariance = scipy.optimize.curve_fit(f = cosineFit, xdata = t, ydata = velSigned, p0 = paramsGuess, bounds = boundVal, full_output = False)
        velMax = paramsFit[0] 
        phi0 = paramsFit[1] 
        T = T_fixed 
    except Exception: 
        velMax = velMax_guess 
        phi0 = phi0_guess 
        T = T_fixed 
        warnings.warn('Cosine fitting failed, using initial parameter estimates') 

In [7]:
def cosinePrep(filepath): 
    T = pd.read_csv(filepath)
    #VarNames = list(T.columns)

    mag = T['Mag'] 
    DMY = pd.to_datetime(T['Date & Time.2']) 

    return mag, DMY

In [8]:
mag, DMY = cosinePrep('C:\\Users\\ehsia\\Downloads\\depthAvg_ADCPdata-new.csv')

In [17]:
extractCosineTideParams(DMY, mag) #actual variable is not mag

C:\Users\ehsia\AppData\Local\Temp\ipykernel_26184\1597940216.py:68: UserWarning: Cosine fitting failed, using initial parameter estimates
  warnings.warn('Cosine fitting failed, using initial parameter estimates')
